# E03 — main model with market-relative and volatility-scaled inputs (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about 15–25 minutes. Keep the tab open.

**One change vs E01, fixed before any run (experiment ledger, E03):** 11 new inputs from `src/features_xs.py`, on top of the 18 price
inputs and the 3 news channels:
- `zr_1/5/10`: past 1/5/10-day return ÷ (trailing 60-day volatility × √h)
- `mrel_r_1/5/10/20`: past return minus the same-day average over SET50 members on that day
- `srel_r_1/5/10/20`: past return minus the same-day average of the *other* members in the same sector (needs ≥ 3 members)

Everything else as E01: `multiscale_sector` + news, lr 1e-4, dropout 0.2, max 30 epochs, patience 10, seeds 20261015/16/17.
XGBoost with the same 29 price inputs runs as a reference (not a tuning attempt).

The look-ahead test (`tests/test_no_lookahead.py`) runs first and must pass.
Outputs in `MyDrive/Senior/results/`: `main_model_e03.json` + `_preds.csv.gz`, `xgb_e03.json` + `_preds.csv.gz`, `results_summary_e03.xlsx`.


In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os
for f in ['src/features_xs.py', 'tests/test_no_lookahead.py', 'scripts/baselines_deep.py', 'scripts/baselines_tabular.py']:
    assert os.path.exists(f), f'stop: push {f} from your PC, then run again'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
assert '--extra-features' in open('scripts/baselines_deep.py', encoding='utf-8').read(), 'stop: push the latest baselines_deep.py'


In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
!python tests/test_pipeline.py

In [ ]:
# Look-ahead test: every input on or before a cut-off must be unchanged when the future is scrambled (about 3 minutes)
!python tests/test_no_lookahead.py | grep -v "defined on"


In [ ]:
# E03: main model, E01 settings + the 11 new inputs
!python scripts/baselines_deep.py --models multiscale_sector --news data/derived/sentiment_scores.csv --extra-features e03 \
    --lr 1e-4 --dropout 0.2 --max-epochs 30 --patience 10 \
    --out "$OUT_DIR/main_model_e03.json"


In [ ]:
# Reference: XGBoost with the same 29 price inputs (CPU, a few minutes)
!python scripts/baselines_tabular.py --extra-features e03 --out "$OUT_DIR/xgb_e03.json"


In [ ]:
# Quick look: best epoch per seed and validation macro-F1 against the pass bar
import json, statistics as st
r = json.load(open(f'{OUT_DIR}/main_model_e03.json'))
print({k: r['settings'][k] for k in ('lr', 'dropout', 'max_epochs', 'patience')}, len(r['settings']['channels']), 'channels', r['git_commit'])
runs = r['models']['multiscale_sector']['runs']
for run in runs:
    print('seed', run['seed'], 'best epoch', run['best_epoch'], 'of', len(run['history']),
          {h: round(run['val'][h]['macro_f1'], 3) for h in ('h1', 'h5', 'h10')})
bar = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
x = json.load(open(f'{OUT_DIR}/xgb_e03.json'))
for h in bar:
    m = st.mean(run['val'][h]['macro_f1'] for run in runs)
    print(h, 'E03', round(m, 4), '| XGBoost+E03', x['results'][h]['model']['xgb_trend']['macro_f1']['mean'], '| bar', bar[h])


In [ ]:
# Side-by-side spreadsheet: E01 vs E03
!python scripts/summarize_results.py --inputs "$OUT_DIR/main_model_v1_news_p41.json" "$OUT_DIR/main_model_e03.json" --out "$OUT_DIR/results_summary_e03.xlsx"
